# Plant prediction to 3D Bounding Box

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from torch.distributed.pipelining import pipeline
from tqdm import tqdm
from ultralytics import YOLO

from plantdb.commons.test_database import test_database

In [ ]:
db = test_database()
db.connect()

In [ ]:
db.list_scans()

## RGB image

In [ ]:
scan_dataset = db.get_scan('real_plant_analyzed')
image_fileset = scan_dataset.get_fileset('images')

In [ ]:
images_path = sorted([image_fileset.get_file(f).path() for f in image_fileset.list_files()])

## Plant detection with YOLO11

In [ ]:
model = YOLO(f"{db.path()}/models/models/yolo11l.pt", 'detect')

In [ ]:
from PIL import Image
import plotly.express as px

images = []
for img_id in tqdm(image_fileset.list_files()):
    img = image_fileset.get_file(img_id)
    results = model(img.path(), verbose=False)

    if not results:
      continue

    # take the first result that contains a box (simplification)
    r = results[0]
    im_bgr = r.plot()                      # BGR numpy array
    im_rgb = Image.fromarray(im_bgr[..., ::-1])  # RGB PIL image
    images.append(im_rgb)

layout_style = {'height': 600, 'width': 800, 'title': "Detection Carousel", 'showlegend': False, 'xaxis': {'visible': False}, 'yaxis': {'visible': False}}

array = np.array([np.array(img) for img in images])
fig = px.imshow(array, animation_frame=0, binary_string=True, labels=dict(animation_frame="Image"))
fig.update_layout(**layout_style)
fig.update_scenes(aspectmode='data')

fig.show()

## Plant detection with YOLO11 - Bounding Box

In [ ]:
confidence_threshold = 0.5
label = 'potted plant'

In [ ]:
detection_results = []

for img_id in tqdm(image_fileset.list_files()):
    img = image_fileset.get_file(img_id)
    img_path = img.path()
    results = model(img_path, verbose=False)

    plant_boxes = []
    if results:
        for r in results:
            for box in r.boxes:
                if r.names[int(box.cls)] == label and box.conf > confidence_threshold:
                    plant_boxes.append(box.xyxy[0].cpu().numpy())

    if not plant_boxes:
        print(f"No plant detected in image {img.id}, skipping...")
        continue

    largest_box = max(plant_boxes, key=lambda b: (b[2] - b[0]) * (b[3] - b[1]))

    # Retrieve camera parameters needed for triangulation
    cam = img.get_metadata('colmap_camera', default=None)
    if cam is None:
        print(f"Could not get camera params from 'colmap_camera' for {img.id}, skipping...")
        continue

    # Store detection info for later triangulation
    detection_results.append({
        "file_id": img.id,
        "box": largest_box,
        "camera": cam,
    })

In [ ]:
if len(detection_results) < 5:
    print("Need at least 5 views with detections to compute a 3D bounding box.")

## Backproject 2D points in 3D

In [ ]:
def get_projection_matrix(cam_meta):
    """
    Computes the projection matrix for a camera based on its metadata.

    This method calculates the camera's projection matrix, which is a
    combination of its intrinsic parameters (describing its optical
    properties) and extrinsic parameters (describing its position and
    orientation in the world). The result is used to map 3D points from
    world space to the camera's image plane.

    Parameters
    ----------
    cam_meta : dict
        A dictionary containing camera metadata. This includes:
            - `camera_model` : dict with key `'params'`, where `params` is a list
              containing intrinsic parameters `[fx, fy, cx, cy]`.
            - `rotmat` : list or array containing the 3x3 rotation matrix for
              the camera.
            - `tvec` : list or array containing the translation vector (size 3).

    Returns
    -------
    numpy.ndarray
        A 3x4 projection matrix, which is the product of the camera's intrinsic
        matrix and its extrinsic transformation (rotation and translation).
    """
    intrinsics_params = cam_meta["camera_model"]['params']
    fx, fy, cx, cy = intrinsics_params[0:4]
    k = np.array([[fx, 0, cx], [0, fy, cy], [0, 0, 1]])  # intrinsic matrix
    rot_mat = np.array(cam_meta['rotmat'])
    tvec = np.array(cam_meta['tvec']).reshape(3, 1)
    return k @ np.hstack((rot_mat, tvec))  # 3x4 projection matrix

In [ ]:
def triangulate_point(p1, p2, m1, m2):
    """
    Triangulate a 3D point from two image projections.

    Parameters
    ----------
    p1 : array_like
        The (u, v) pixel coordinates of the point in the first image.
    p2 : array_like
        The (u, v) pixel coordinates of the point in the second image.
    m1 : array_like, shape (3, 4)
        The projection matrix of the first camera.
    m2 : array_like, shape (3, 4)
        The projection matrix of the second camera.

    Returns
    -------
    p_3d : ndarray
        The 3‑D point in inhomogeneous coordinates, shape (3,).

    Notes
    -----
    The method builds a linear system from the cross‑product constraints
    imposed by each image point.  Singular value decomposition is then
    used to find the null‑space of this system, and the last column of
    the right‑singular matrix gives the homogeneous solution.  The
    homogeneous coordinate is normalised to produce the inhomogeneous
    3‑D point that is returned.
    """
    u1, v1 = p1
    u2, v2 = p2

    # Build linear system A * X = 0 from cross‑product constraints
    a = np.array([
        u1 * m1[2, :] - m1[0, :],
        v1 * m1[2, :] - m1[1, :],
        u2 * m2[2, :] - m2[0, :],
        v2 * m2[2, :] - m2[1, :]
    ])

    _, _, vh = np.linalg.svd(a)
    p_3d_h = vh[-1]  # solution in homogeneous coordinates
    return p_3d_h[:3] / p_3d_h[3]  # convert to inhomogeneous

In [ ]:
def triangulate_all_corners(detection_results):
    """
    Triangulate all corner points from pairs of detection results.

    This method iterates over all unique pairs of detection results, obtains the projection matrices for the cameras involved, and triangulates every corner pair between the two bounding boxes. Any pair for which the projection matrix cannot be retrieved is skipped, with a warning logged. The result is a list of 3‑D points corresponding to the triangulated corners.

    Parameters
    ----------
    detection_results : list[dict]
        A list of detection dictionaries. Each dictionary must contain a ``'camera'`` key used to obtain the projection matrix, a ``'file_id'`` key used for logging, and a ``'box'`` key which is a 4‑tuple or list ``(x_min, y_min, x_max, y_max)`` describing the bounding box of the detection.

    Returns
    -------
    list[np.ndarray]
        A list of 3‑D points (homogeneous coordinates) obtained by triangulating each corner pair. Each point is returned as a NumPy array of shape (4,).

    Notes
    -----
    If the projection matrix for a camera cannot be retrieved due to missing data or an invalid format, that camera pair is skipped and a warning is logged. The function does not raise exceptions for such cases; it continues processing the remaining pairs.
    """
    import itertools

    all_3d_points = []

    for r1, r2 in itertools.combinations(detection_results, 2):
        try:
            m1 = get_projection_matrix(r1['camera'])
            m2 = get_projection_matrix(r2['camera'])
        except (KeyError, TypeError, ValueError) as e:
            print(f"Could not get projection matrix for a camera, "
                  f"skipping pair ({r1['file_id']}, {r2['file_id']}). Error: {e}")
            continue

        b1 = r1['box']
        b2 = r2['box']
        # Extract the four corners of each bounding box
        corners1 = [(b1[0], b1[1]), (b1[2], b1[1]), (b1[0], b1[3]), (b1[2], b1[3])]
        corners2 = [(b2[0], b2[1]), (b2[2], b2[1]), (b2[0], b2[3]), (b2[2], b2[3])]

        for p1 in corners1:
            for p2 in corners2:
                point_3d = triangulate_point(p1, p2, m1, m2)
                all_3d_points.append(point_3d)

    return all_3d_points


In [ ]:
all_3d_points = triangulate_all_corners(detection_results)

In [ ]:
points = np.array(all_3d_points)
print(points.shape)

In [ ]:
min_coords = points.min(axis=0)
max_coords = points.max(axis=0)

In [ ]:
bbox_3d = {
    "x": [float(min_coords[0]), float(max_coords[0])],
    "y": [float(min_coords[1]), float(max_coords[1])],
    "z": [float(min_coords[2]), float(max_coords[2])],
}

In [ ]:
bbox_3d

## Distance based filtering

### Convergence heuristic

In [ ]:
import copy as cp
from scipy.spatial import distance_matrix

def points_pw_max_dist(points):
    # Pairwise distance matrix: shape (N, N)
    dist_mat = distance_matrix(points, points)
    # For each point, compute its minimal distance to another (exclude self, so set diagonal to np.inf)
    np.fill_diagonal(dist_mat, 0)
    return dist_mat.max(axis=1)

filtered_points = cp.deepcopy(points)

max_distances = points_pw_max_dist(filtered_points)
max_th = np.percentile(max_distances, 95)
iter = 1
while np.sum(max_distances > max_th) > 5:
    print(f"Got {np.sum(max_distances > max_th)} far points at iteration {iter} with a threshold of {max_th}...")
    keep_mask = max_distances < max_th
    filtered_points = filtered_points[keep_mask]
    max_distances = points_pw_max_dist(filtered_points)
    max_th = np.percentile(max_distances, 95)
    iter += 1

In [ ]:
print(f"Original: {len(points)} points → Filtered: {len(filtered_points)} points")

In [ ]:
plt.figure(figsize=(12, 3))
plt.boxplot(max_distances, vert=False)
plt.show()

### KDTree

In [ ]:
from typing import Tuple
from scipy.spatial import KDTree

def filter_isolated(points: np.ndarray,
                    k: int = 5,
                    percentile: float = 95.0) -> Tuple[np.ndarray, np.ndarray]:
    """
    Remove points that are isolated with respect to the rest of the dataset.

    Parameters
    ----------
    points : (N, D) array_like
        Input coordinates.
    k : int, default 5
        Number of neighbors to consider (excluding the point itself).
    percentile : float, default 95.0
        Percentile of the k‑th nearest‑neighbor distance that defines the
        isolation threshold.

    Returns
    -------
    numpy.ndarray
        An (M, D) array of points that are not considered isolated.
    numpy.ndarray
        A vector of size M giving the distance of the non-isolated points.
    """
    tree = KDTree(points)
    # Query k+1 because the first neighbor is the point itself (distance 0)
    distances, _ = tree.query(points, k=k + 1, p=2, workers=-1)
    # Distance to the k‑th nearest neighbor (index 1 is the first real neighbor)
    kth_dist = distances[:, k] if k < distances.shape[1] else distances[:, -1]

    # Threshold: keep points whose k‑th neighbor distance is below the
    # specified percentile of the overall distribution
    thresh = np.percentile(kth_dist, percentile)
    keep = kth_dist <= thresh
    return points[keep], kth_dist[keep]

In [ ]:
filtered_points, pts_distances = filter_isolated(points, k=10, percentile=98)

In [ ]:
print(f"Original: {len(points)} points → Filtered: {len(filtered_points)} points")

In [ ]:
plt.figure(figsize=(12, 3))
plt.boxplot(pts_distances, vert=False)
plt.show()

### DBSCAN

In [ ]:
from sklearn.cluster import DBSCAN

# eps: neighborhood radius; min_samples: minimal cluster size
db = DBSCAN(eps=2, min_samples=5, metric='euclidean')
labels = db.fit_predict(points)

# Noise points are labeled -1
filtered_points = points[labels != -1]

In [ ]:
print(f"Original: {len(points)} points → Filtered: {len(filtered_points)} points")

In [ ]:
max_distances = points_pw_max_dist(filtered_points)

In [ ]:
plt.figure(figsize=(12, 3))
plt.boxplot(max_distances, vert=False)
plt.show()

## 3D points clustering

In [ ]:
import numpy as np
from sklearn.cluster import KMeans

def cluster_3d_points(points, n_clusters=4):
    """
    Cluster 3D points into 4 groups using Euclidean distance.

    Parameters:
    all_3d_points (numpy.ndarray): Nx3 array of 3D points.
    n_clusters (int): Number of clusters (default is 4).

    Returns:
    numpy.ndarray: Array of cluster labels for each point.
    """
    # Initialize KMeans with 4 clusters and Euclidean distance
    kmeans = KMeans(n_clusters=n_clusters, random_state=42)

    # Fit the model to the data
    labels = kmeans.fit_predict(points)
    # Return the cluster labels
    return labels

In [ ]:
labels = cluster_3d_points(filtered_points)
print(labels)


In [ ]:
import plotly.graph_objects as go

fig = go.Figure(
    data=go.Scatter3d(
        x=filtered_points[:, 0],          # X‑coordinates
        y=filtered_points[:, 1],          # Y‑coordinates
        z=filtered_points[:, 2],          # Z‑coordinates
        mode='markers',
        marker=dict(size=5, color=labels),
    )
)

fig.update_layout(
    title='3‑D Scatter of Backprojected Points',
    scene=dict(
        xaxis_title='X (mm)',
        yaxis_title='Y (mm)',
        zaxis_title='Z (mm)'
    )
)

fig.show()

In [ ]:
min_coords = filtered_points.min(axis=0)
max_coords = filtered_points.max(axis=0)

In [ ]:
bbox_3d = {
    "x": [float(min_coords[0]), float(max_coords[0])],
    "y": [float(min_coords[1]), float(max_coords[1])],
    "z": [float(min_coords[2]), float(max_coords[2])],
}

In [ ]:
bbox_3d

In [ ]:
import toml

cfg_toml = scan_dataset.path() / "pipeline.toml"
cfg = toml.load(cfg_toml)

cfg["Voxels"]['bounding_box']